# Can AI See Better in 3D? Text-prompted SAM on UNT campus (A: photo, B: LiDAR, C: fusion)

Runs **Grounding DINO + Segment Anything** (the same idea as ArcGIS *Text SAM*) with the text prompt **"building"** on three input images, then scores each result against the LiDAR building ground truth.

**Before you start**
1. *Runtime → Change runtime type → **T4 GPU*** → Save.
2. In Google Drive, create a folder **`GIS_Day_SAM`** and upload these 4 files from `D:\GIS_Day\Lidar\`:
   `A_photo_export.tif`, `B_lidar_export.tif`, `C_fusion_export.tif`, `bldg_truth_export.tif`
3. Run the cells **one by one, top to bottom** (click ▶ on each, wait for it to finish).


## 1. Check the GPU
You should see a **Tesla T4** in the output. If it says no GPU, go back to step 1 above.

In [ ]:
!nvidia-smi

## 2. Install the software (about 3–5 minutes)
Ignore red 'dependency' warnings as long as the cell finishes. If Colab asks you to **restart the session**, click Restart and continue with step 3.

In [ ]:
!pip install -q segment-geospatial groundingdino-py rasterio geopandas

## 3. Connect Google Drive and check the 4 files

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DATA = '/content/drive/MyDrive/GIS_Day_SAM'
OUT  = os.path.join(DATA, 'results')
os.makedirs(OUT, exist_ok=True)

for f in ['A_photo_export.tif', 'B_lidar_export.tif', 'C_fusion_export.tif', 'bldg_truth_export.tif']:
    p = os.path.join(DATA, f)
    print(('OK      ' if os.path.exists(p) else 'MISSING ') + p)

## 4. Look at the inputs (bands, size, pixel type)
Each input should have **3 bands**, type **uint8**, and the same width/height.

In [ ]:
import rasterio
for f in ['A_photo_export.tif', 'B_lidar_export.tif', 'C_fusion_export.tif', 'bldg_truth_export.tif']:
    with rasterio.open(os.path.join(DATA, f)) as r:
        print(f'{f:15s} bands={r.count} size={r.width}x{r.height} dtype={r.dtypes[0]} cell={r.res[0]:.2f} m crs={r.crs}')

## 5. Settings — identical for A, B and C (controlled experiment)
These match the ArcGIS Text SAM defaults you used. **Do not change them between runs.**

In [ ]:
TEXT_PROMPT    = 'building'
BOX_THRESHOLD  = 0.2
TEXT_THRESHOLD = 0.2
TILE = 1024      # tile size in pixels (512 m at 0.5 m cells)
STEP = 896       # 128-pixel overlap between tiles so buildings on tile edges are not cut

## 6. Load the model (downloads ~2–3 GB the first time)

In [ ]:
from samgeo.text_sam import LangSAM
sam = LangSAM()
print('Model loaded')

## 7. The detection function
Splits the image into overlapping tiles, runs text-prompted SAM on each tile, merges the masks (a pixel is *building* if any tile says so), and converts the mask to polygons.

In [ ]:
import numpy as np, glob, shutil
from rasterio.windows import Window
from rasterio.merge import merge
from rasterio import features
import geopandas as gpd
from shapely.geometry import shape

def run_sam(name):
    src_path = os.path.join(DATA, f'{name}.tif')
    work = f'/content/work_{name}'
    shutil.rmtree(work, ignore_errors=True)
    os.makedirs(f'{work}/tiles'); os.makedirs(f'{work}/masks')

    # 1) tile the image
    with rasterio.open(src_path) as src:
        prof = src.profile
        tiles = []
        for row in range(0, src.height, STEP):
            for col in range(0, src.width, STEP):
                w = Window(col, row, min(TILE, src.width - col), min(TILE, src.height - row))
                if w.width < 64 or w.height < 64:
                    continue
                p = prof.copy(); p.update(width=w.width, height=w.height, transform=src.window_transform(w), compress='lzw')
                tp = f'{work}/tiles/t_{row}_{col}.tif'
                with rasterio.open(tp, 'w', **p) as dst:
                    dst.write(src.read(window=w))
                tiles.append(tp)
    print(f'{name}: {len(tiles)} tiles')

    # 2) text-prompted SAM on each tile
    masks = []
    for i, tp in enumerate(tiles):
        mp = tp.replace('/tiles/', '/masks/')
        try:
            sam.predict(tp, TEXT_PROMPT, box_threshold=BOX_THRESHOLD, text_threshold=TEXT_THRESHOLD, output=mp)
        except Exception as e:
            print('  tile', i, 'no detection / error:', str(e)[:80])
        if not os.path.exists(mp):          # nothing found -> write an empty mask
            with rasterio.open(tp) as t:
                p = t.profile; p.update(count=1, dtype='uint8')
                with rasterio.open(mp, 'w', **p) as d:
                    d.write(np.zeros((1, t.height, t.width), 'uint8'))
        masks.append(mp)
        print(f'  tile {i+1}/{len(tiles)} done')

    # 3) merge tile masks (max = building if any tile says building)
    srcs = [rasterio.open(m) for m in masks]
    mosaic, tr = merge(srcs, method='max')
    for s in srcs: s.close()
    mosaic = (mosaic[0] > 0).astype('uint8')
    p = prof.copy(); p.update(count=1, dtype='uint8', transform=tr, width=mosaic.shape[1], height=mosaic.shape[0], compress='lzw', nodata=None)
    mask_path = os.path.join(OUT, f'sam_{name}_mask.tif')
    with rasterio.open(mask_path, 'w', **p) as d:
        d.write(mosaic, 1)

    # 4) polygons
    geoms = [shape(g) for g, v in features.shapes(mosaic, mask=mosaic == 1, transform=tr) if v == 1]
    gdf = gpd.GeoDataFrame({'Class': ['building'] * len(geoms)}, geometry=geoms, crs=prof['crs'])
    gdf['area_m2'] = gdf.area
    gdf = gdf[gdf.area_m2 >= 10]            # drop specks smaller than 10 m2
    shp = os.path.join(OUT, f'sam_{name}.shp')
    gdf.to_file(shp)
    print(f'{name}: {len(gdf)} building polygons, {mosaic.mean()*100:.1f}% of area -> {shp}')
    return mask_path

## 8. Run it on A, B and C (a few minutes each)

In [ ]:
mask_A = run_sam('A_photo_export')

In [ ]:
mask_B = run_sam('B_lidar_export')

In [ ]:
mask_C = run_sam('C_fusion_export')

## 9. Accuracy against the LiDAR ground truth
Pixel-by-pixel comparison with `bldg_truth.tif` (1 = building from LiDAR class 6).

- **Precision**: of what SAM called building, how much really is? (low = false alarms on trees/parking)
- **Recall**: of all real buildings, how much did SAM find? (low = missed buildings)
- **F1**: balance of both — **the main score**
- **IoU**: overlap ÷ union of the two building maps

In [ ]:
import pandas as pd
from rasterio.warp import reproject, Resampling

def score(mask_path, label):
    with rasterio.open(mask_path) as m:
        pred = m.read(1) > 0
        truth = np.zeros(pred.shape, 'uint8')
        with rasterio.open(os.path.join(DATA, 'bldg_truth_export.tif')) as t:
            reproject(t.read(1), truth, src_transform=t.transform, src_crs=t.crs,
                      dst_transform=m.transform, dst_crs=m.crs, resampling=Resampling.nearest)
    truth = truth == 1
    tp = np.sum(pred & truth); fp = np.sum(pred & ~truth); fn = np.sum(~pred & truth)
    precision = tp / (tp + fp) if tp + fp else 0
    recall    = tp / (tp + fn) if tp + fn else 0
    f1  = 2 * precision * recall / (precision + recall) if precision + recall else 0
    iou = tp / (tp + fp + fn) if tp + fp + fn else 0
    return dict(Input=label, Precision=round(precision, 3), Recall=round(recall, 3), F1=round(f1, 3), IoU=round(iou, 3),
                TP_m2=int(tp * 0.25), FP_m2=int(fp * 0.25), FN_m2=int(fn * 0.25))

results = pd.DataFrame([score(mask_A, 'A: photo (RGB)'),
                        score(mask_B, 'B: LiDAR only'),
                        score(mask_C, 'C: fusion (NIR+Red+Height)')])
results.to_csv(os.path.join(OUT, 'accuracy_results.csv'), index=False)
results

## 10. Quick bar chart of F1

In [ ]:
import matplotlib.pyplot as plt
ax = results.plot.bar(x='Input', y=['Precision', 'Recall', 'F1'], figsize=(8, 4), rot=0)
ax.set_ylim(0, 1); ax.set_ylabel('score'); ax.set_title('SAM building detection on UNT campus')
plt.tight_layout(); plt.savefig(os.path.join(OUT, 'accuracy_chart.png'), dpi=200); plt.show()

## 11. Done
Everything is saved in Google Drive → `GIS_Day_SAM/results/`:
`sam_A_photo.shp`, `sam_B_lidar.shp`, `sam_C_fusion.shp` (+ masks), `accuracy_results.csv`, `accuracy_chart.png`.

Download the `results` folder and add the shapefiles to ArcGIS Pro to make your maps.